# EDA of Machine Downtime Data 

In [ ]:
#! pip install pandas numpy matplotlib seaborn scipy  psycopg2 



In [ ]:
import psycopg2
import psycopg2.extras               #Package for DB connect with postgre sql
import pandas as pd                  # Data manipulation and analysis
import numpy as np                   # Numerical computing
import matplotlib.pyplot as plt      # Data visualization
import seaborn as sns                # Enhanced data visualization
import scipy.stats as stats          # Statistical functions



### Esatblish Connection with POSTGRES sql and Import Raw data

In [ ]:
#Establish a connection to PostgreSQL database
conn = psycopg2.connect(
     host = "localhost",
     port = "5432",
     database = "postgres",
     user = "postgres",
     password = "**********"
)
cur = conn.cursor(cursor_factory = psycopg2.extras.DictCursor)



In [ ]:
cur.execute("SELECT * FROM machine_downtime")
# Fetch all rows from the cursor
rows = cur.fetchall()

# Get the column names from the cursor description
columns = [desc[0] for desc in cur.description]

# Create a DataFrame from the fetched data
df = pd.DataFrame(rows, columns=columns)

# Close the cursor and connection
cur.close()
conn.close()


In [ ]:
df.head()

In [ ]:



# save the file as csv for future reference
df.to_csv('output.csv', index=False)

In [ ]:
# import the csv file as 'md'(machine_downtime)
md = pd.read_csv('output.csv')

### Data Understanding

In [ ]:
md.head()

In [ ]:
md.tail()

In [ ]:
md.shape

In [ ]:
md.info()

In [ ]:
md.dtypes

In [ ]:
md.columns

In [ ]:
# check null values if any
null_values = md.isnull()
null_count = null_values.sum()
print(null_count)

In [ ]:
md.describe()

### Data Cleansing

In [ ]:
# Fill null values with constant value
#df.fillna(value = 0, inplace=True)

# Fill null values with mean of the column
md.fillna(md.median(numeric_only = True), inplace =True)

# Interpolate null values 
#df.interpolate(inplace = True)

In [ ]:
# Convert date column 'datetime' format
md['date']= pd.to_datetime(md['date'])


#verify the date  column
print(md.dtypes['date'])

In [ ]:
# Convert 'machine_failure' column to Categorical
md['machine_failure'] = md['machine_failure'].astype('category')

# verify the data type
print(md.dtypes['machine_failure'])

In [ ]:
#Verify if any null values exist
md.isna().sum()

In [ ]:
# Checking for duplicate values if any
md.duplicated().sum()

### Measures of Central Tendency

In [ ]:
# Mode for 'machine_failure ' coulmn which is categorical data type and 'Downtime'
mode_value =md[['machine_failure','downtime']].mode()
print(mode_value)

In [ ]:
md.describe()

### Measures of Dispersion

In [ ]:

# Select the columns for dispersion calculation
columns_to_analyze = ['hydraulic_pressure', 'coolant_pressure', 'air_system_pressure', 'coolant_temperature', 'hydraulic_oil_temperature', 'spindle_bearing_temperature', 'spindle_vibration', 'tool_vibration', 'spindle_speed', 'voltage', 'torque', 'cutting']

# Calculate measures of dispersion for the columns
dispersion_measures = pd.DataFrame({
    'min': md[columns_to_analyze].min(),
    'max': md[columns_to_analyze].max(),
    'std': md[columns_to_analyze].std(),
    'var': md[columns_to_analyze].var(),
    'mad': md[columns_to_analyze].mad(),
    'range': md[columns_to_analyze].max() - md[columns_to_analyze].min()
})

# Print the dispersion measures
print(dispersion_measures)



### Outlier Detection

In [ ]:
# Select the columns for outlier detection
columns_to_analyze = ['hydraulic_pressure', 'coolant_pressure', 'air_system_pressure',
                      'coolant_temperature', 'hydraulic_oil_temperature', 'spindle_bearing_temperature',
                      'spindle_vibration', 'tool_vibration', 'spindle_speed', 'voltage', 'torque', 'cutting']

# Create box plots for outlier detection
md[columns_to_analyze].boxplot()
plt.xticks(rotation=60)
plt.title('Box Plots - Outlier Detection')
plt.show()

In [ ]:
# Set the matplotlib backend to display inline
%matplotlib inline

# Create scatter plots for outlier detection
for column in columns_to_analyze:
    plt.scatter(md[column], md[column], c='blue', label='Data')
    
    # Identify outliers and change their color to red
    outliers = md[column].loc[md[column] > md[column].mean() + 2 * md[column].std()]
    plt.scatter(outliers, outliers, c='red', label='Outliers')
    
    plt.xlabel(column)
    plt.ylabel(column)
    plt.title(f'Scatter Plot - {column} (Outlier Detection)')
    plt.legend()
    plt.show()





### Correlation of Variables

In [ ]:
column_corr = md.corr(numeric_only = True)
print("Correlation of Columns :")
print(column_corr)

In [ ]:
 #Set the matplotlib backend to display inline
%matplotlib inline

# Create the heatmap with color scheme
sns.heatmap(column_corr, cmap='RdYlGn', xticklabels=column_corr.columns, yticklabels=column_corr.columns)
plt.title('Correlation Heatmap')
plt.show()

### Skewness

In [ ]:
skewness=md.skew(numeric_only=True)
print(skewness)

In [ ]:
%matplotlib inline

# Create a bar plot of skewness
sns.barplot(x=skewness.index, y=skewness.values)
plt.xticks(rotation=75)
plt.xlabel('Columns')
plt.ylabel('Skewness')
plt.title('Skewness of Columns')
plt.show()

In [ ]:
for column in columns_to_analyze:
  plt.figure()
  sns.histplot(data=md, x=column, kde=True)
  plt.xlabel(column)
  plt.ylabel('Density')
  plt.title(f'Skewness: {skewness[column]:.2f}')
  plt.show()


### Kurtosis of Variables

In [ ]:
# Calculate the kurtosis for each column
kurtosis_values = md.kurtosis(numeric_only = True)
print(kurtosis_values)

In [ ]:
%matplotlib inline

# Create a bar plot of kurtosis
sns.barplot(x=kurtosis_values.index, y=kurtosis_values.values)
plt.xticks(rotation=75)
plt.xlabel('Columns')
plt.ylabel('Kurtosis')
plt.title('Kurtosis of Columns')
plt.show()

1)What is the proportion of machine downtime occurrences in the dataset?

In [ ]:
# Count the occurrences of machine failure
failure_counts = md['machine_failure'].value_counts()

# Plot a pie chart
plt.figure(figsize=(8, 8))
plt.pie(failure_counts, labels=failure_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Machine Failure Occurrences')
plt.axis('equal')
plt.show()

 2)What is the distribution of downtime occurrences across different dates? Are there any specific periods or trends where downtime is more prevalent?

In [ ]:

machine_downtime_counts  = md[(md.machine_failure =="Yes")]

In [ ]:
machine_downtime_counts

In [ ]:
mdc = machine_downtime_counts.groupby('date')['machine_failure'].count()

In [ ]:
# Plot the distribution of downtime occurrences over time
plt.figure(figsize=(20, 10))
plt.plot(mdc.index,mdc.values)
plt.xlabel('Date')
plt.ylabel('Downtime Occurrences')
plt.title('Distribution of Downtime Occurrences over Time')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Plot the histogram by date
sns.histplot(data=mdc, x='date', bins=10)
plt.xlabel('Date')
plt.ylabel('Frequency')
plt.title('Histogram of Downtime Occurrences by Date')
plt.xticks(rotation=45)
plt.show()

3)Which machine(s) (identified by Machine_ID) experience the highest frequency or duration of unplanned downtime?

In [ ]:
mdc_mid = machine_downtime_counts.groupby('machine_id')['machine_failure'].count()

In [ ]:
mdc_mid

In [ ]:
# Plot the pie chart
plt.figure(figsize=(6, 6))
plt.pie(mdc_mid, labels=mdc_mid.index, autopct='%1.1f%%')
plt.title('Downtime Occurrences by Machine ID')
plt.show()

4)Are there any noticeable differences in downtime patterns between different assembly lines (Assembly_Line_No)?

In [ ]:
mdc_malno = machine_downtime_counts.groupby('assembly_line_no')['machine_failure'].count()

In [ ]:
mdc_malno

In [ ]:
# Plot the pie chart
plt.figure(figsize=(6, 6))
plt.pie(mdc_malno, labels=mdc_malno.index, autopct='%1.1f%%')
plt.title('Downtime Occurrences by Assembly Line No')
plt.show()

In [ ]:
# Plot the downtime occurrences by assembly line
plt.figure(figsize=(10, 6))
mdc_malno.plot(kind='bar')
plt.xlabel('Assembly Line No')
plt.ylabel('Downtime Occurrences')
plt.title('Downtime Occurrences by Assembly Line')
plt.show()

5)Can we identify specific temperature or vibration thresholds (Coolant Temperature, Spindle Vibration, Tool Vibration) that are associated with increased downtime?

In [ ]:


# Scatter plot of coolant temperature vs. machine failure occurrences
plt.figure(figsize=(10,4))
sns.scatterplot(data=md, x='coolant_temperature', y='machine_failure', hue='machine_failure')
plt.xlabel('Coolant Temperature')
plt.ylabel('Machine Failure')
plt.title('Coolant Temperature vs. Machine Failure Occurrences')
plt.show()


# Scatter plot of spindle vibration vs. machine failure occurrences
plt.figure(figsize=(10, 6))
sns.scatterplot(data=md, x='spindle_vibration', y='machine_failure', hue='machine_failure')
plt.xlabel('Spindle Vibration')
plt.ylabel('Machine Failure')
plt.title('Spindle Vibration vs. Machine Failure Occurrences')
plt.show()

# Scatter plot of tool vibration vs. machine failure occurrences
plt.figure(figsize=(10, 6))
sns.scatterplot(data=md, x='tool_vibration', y='machine_failure', hue='machine_failure')
plt.xlabel('Tool Vibration')
plt.ylabel('Machine Failure')
plt.title('Tool Vibration vs. Machine Failure Occurrences')
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.barplot( x= 'machine_id', y  ='spindle_speed', data = md, hue ='machine_failure')
plt.title("Machine Failure by Spindle Speed ")
plt.show()